# Versión del script para entrenar el modelo sin los rezagos de dengue

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.model_selection import TimeSeriesSplit
import optuna

# ==================== CONFIGURACIÓN ====================
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
processed_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Transformar el target con log1p (ayuda con la fuerte asimetría/picos de casos_dengue).
# Poner en False si se prefiere trabajar en la escala original.
USE_LOG_TARGET = True

# ==================== CARGA DE DATOS ====================
print("Cargando datos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']

base_meteo_vars = [
    'temp', 'temp_max', 'temp_min',
    'hum_esp', 'hum_rel',
    'prec', 'dias_lluvia',
    'vel_vi', 'vel_vi_max', 'vel_vi_min',
    'soi', 'sst'
]
base_meteo_vars = [v for v in base_meteo_vars if v in df.columns]

lag_predictor_cols = []
for var in base_meteo_vars:
    for lag in range(1, 13):
        col = f'{var}_lag_{lag}'
        if col in df.columns:
            lag_predictor_cols.append(col)

print(f"Variables meteo base: {len(base_meteo_vars)}")
print(f"Rezagos meteo disponibles (1-12): {len(lag_predictor_cols)}")

# ==================== INGENIERÍA DE ATRIBUTOS CLIMÁTICOS ====================
# Todo lo que sigue se construye SOLO a partir de variables meteorológicas/climáticas
# (crudas o sus rezagos ya existentes). Nunca se usa casos_dengue ni sus rezagos.
print("\nCreando variables climáticas derivadas...")

df_eng = df.copy()
train_years_for_clima = (df_eng['año'] >= 2021) & (df_eng['año'] <= 2025)

engineered_cols = []

# --- A. Anomalías estacionales ---
# Se compara cada valor con el promedio histórico de esa misma semana epidemiológica,
# calculado SOLO con los años de entrenamiento (2021-2025), para no filtrar información
# del año de test (2026) hacia atrás. Esto separa el efecto "año atípico" del ciclo
# estacional normal, que se pierde al usar solo el valor crudo.
for var in base_meteo_vars:
    clima = (
        df_eng.loc[train_years_for_clima]
        .groupby('semana_epi')[var]
        .mean()
    )
    media_global = df_eng.loc[train_years_for_clima, var].mean()
    col_clima = f'{var}_clima_semana'
    col_anom = f'{var}_anomalia'
    df_eng[col_clima] = df_eng['semana_epi'].map(clima).fillna(media_global)
    df_eng[col_anom] = df_eng[var] - df_eng[col_clima]
    engineered_cols.append(col_anom)
    # La climatología en sí también puede aportar señal estacional pura
    engineered_cols.append(col_clima)

# --- B. Acumulados (lluvia y días de lluvia) ---
# La acumulación de lluvia en semanas recientes suele ser más relevante para el ciclo
# del vector que el valor puntual de una sola semana.
if 'prec' in df_eng.columns:
    for window in [4, 8, 12]:
        col = f'prec_acum_{window}'
        df_eng[col] = df_eng['prec'].rolling(window=window, min_periods=1).sum()
        engineered_cols.append(col)

if 'dias_lluvia' in df_eng.columns:
    for window in [4, 8, 12]:
        col = f'dias_lluvia_acum_{window}'
        df_eng[col] = df_eng['dias_lluvia'].rolling(window=window, min_periods=1).sum()
        engineered_cols.append(col)

# --- C. Rachas de condiciones favorables ---
# Número de semanas consecutivas con temperatura (o lluvia) por encima de su propio
# promedio climatológico estacional: proxy simple de "condiciones sostenidas" favorables
# para la proliferación del vector.
def racha_consecutiva(condicion):
    racha = np.zeros(len(condicion), dtype=int)
    contador = 0
    for i, val in enumerate(condicion):
        if val:
            contador += 1
        else:
            contador = 0
        racha[i] = contador
    return racha

if 'temp_anomalia' in df_eng.columns:
    df_eng['racha_temp_alta'] = racha_consecutiva((df_eng['temp_anomalia'] > 0).values)
    engineered_cols.append('racha_temp_alta')

if 'prec_anomalia' in df_eng.columns:
    df_eng['racha_prec_alta'] = racha_consecutiva((df_eng['prec_anomalia'] > 0).values)
    engineered_cols.append('racha_prec_alta')

# --- D. Interacciones entre variables meteorológicas ---
# Combinaciones simples que aproximan condiciones conjuntas de temperatura/humedad
# favorables para el mosquito vector, sin involucrar casos_dengue en ningún término.
if set(['temp', 'hum_rel']).issubset(df_eng.columns):
    df_eng['temp_x_hum_rel'] = df_eng['temp'] * df_eng['hum_rel']
    engineered_cols.append('temp_x_hum_rel')

if set(['temp_max', 'prec']).issubset(df_eng.columns):
    df_eng['temp_max_x_prec'] = df_eng['temp_max'] * df_eng['prec']
    engineered_cols.append('temp_max_x_prec')

if set(['temp', 'hum_rel']).issubset(df_eng.columns):
    # Índice simplificado tipo "capacidad vectorial": mayor con temperatura y humedad altas.
    df_eng['indice_vectorial'] = (df_eng['temp'] * df_eng['hum_rel']) / 100.0
    engineered_cols.append('indice_vectorial')

# --- E. Agregados de rezagos largos (SOI / SST) ---
# El dataset solo trae rezagos hasta 12 semanas, así que en vez de "inventar" rezagos
# más largos que no existen, se resumen los ya disponibles (promedio, mínimo, máximo
# de los últimos 6 y 12 rezagos) para capturar la tendencia climática de fondo
# (El Niño / La Niña), que actúa con retardos largos.
for var in ['soi', 'sst']:
    lag_cols_var = [f'{var}_lag_{lag}' for lag in range(1, 13) if f'{var}_lag_{lag}' in df_eng.columns]
    lag_cols_6 = [c for c in lag_cols_var if int(c.split('_')[-1]) <= 6]
    lag_cols_12 = lag_cols_var
    if lag_cols_6:
        df_eng[f'{var}_avg_6'] = df_eng[lag_cols_6].mean(axis=1)
        df_eng[f'{var}_min_6'] = df_eng[lag_cols_6].min(axis=1)
        df_eng[f'{var}_max_6'] = df_eng[lag_cols_6].max(axis=1)
        engineered_cols += [f'{var}_avg_6', f'{var}_min_6', f'{var}_max_6']
    if lag_cols_12:
        df_eng[f'{var}_avg_12'] = df_eng[lag_cols_12].mean(axis=1)
        df_eng[f'{var}_min_12'] = df_eng[lag_cols_12].min(axis=1)
        df_eng[f'{var}_max_12'] = df_eng[lag_cols_12].max(axis=1)
        engineered_cols += [f'{var}_avg_12', f'{var}_min_12', f'{var}_max_12']

engineered_cols = list(dict.fromkeys(engineered_cols))  # quitar duplicados preservando orden
print(f"Variables climáticas derivadas creadas: {len(engineered_cols)}")

# ==================== CONSOLIDAR PREDICTORES PERMITIDOS ====================
predictor_cols = base_meteo_vars + lag_predictor_cols + engineered_cols
predictor_cols = [c for c in predictor_cols if c in df_eng.columns]
print(f"\nTotal de predictores candidatos (meteo + rezagos + derivadas, sin casos_dengue): {len(predictor_cols)}")

df_model = df_eng[exclude_cols + [target_col] + predictor_cols].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan)
df_model = df_model.dropna(subset=predictor_cols + [target_col]).reset_index(drop=True)
print(f"Registros disponibles tras eliminar NAs/infinitos: {len(df_model)}")

# ==================== SELECCIÓN DE CARACTERÍSTICAS CON RANDOM FOREST ====================
print("\nSeleccionando características con Random Forest...")

X_rf = df_model[predictor_cols].values
y_rf = df_model[target_col].values

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_rf, y_rf)

feature_importance = pd.DataFrame({
    'Feature': predictor_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

n_selected = min(30, len(predictor_cols))
selected_features_rf = feature_importance.head(n_selected)['Feature'].tolist()

print(f"Top {n_selected} características seleccionadas por RF:")
for i in range(min(15, len(selected_features_rf))):
    print(f"  {i+1}. {feature_importance.loc[i, 'Feature']}: {feature_importance.loc[i, 'Importance']:.4f}")

features_file = os.path.join(output_dir, 'atributos_seleccionados_rf.xlsx')
feature_importance.to_excel(features_file, index=False)
print(f"\nLista de atributos seleccionados guardada en: {features_file}")

# ==================== PREPARACIÓN DE DATOS FINALES ====================
print("\nPreparando datos finales...")

X = df_model[selected_features_rf].values
y_original = df_model[target_col].values
y = np.log1p(y_original) if USE_LOG_TARGET else y_original.copy()

fechas = df_model['fecha'].values
años = df_model['año'].values
semanas = df_model['semana_epi'].values

train_mask = (años >= 2021) & (años <= 2025)
test_mask = años == 2026

X_train, y_train, y_train_orig = X[train_mask], y[train_mask], y_original[train_mask]
X_test, y_test, y_test_orig = X[test_mask], y[test_mask], y_original[test_mask]
fechas_train, fechas_test = fechas[train_mask], fechas[test_mask]
semanas_train, semanas_test = semanas[train_mask], semanas[test_mask]

print(f"Entrenamiento: {len(X_train)} registros")
print(f"Test: {len(X_test)} registros")

# ==================== PESOS SUAVIZADOS PARA PICOS ====================
# Respecto a la versión anterior, se reducen los factores extremos (antes hasta 5x)
# para no forzar al modelo a una solución demasiado conservadora en la optimización.
def create_soft_weights(y, peak_threshold=0.80, extreme_threshold=0.95):
    weights = np.ones_like(y, dtype=float)
    threshold_high = np.percentile(y, peak_threshold * 100)
    threshold_extreme = np.percentile(y, extreme_threshold * 100)

    for i, val in enumerate(y):
        if val >= threshold_extreme:
            weight_factor = 1.8 + (val - threshold_extreme) / (threshold_extreme + 1) * 0.7  # máx ~2.5x
            weights[i] *= weight_factor
        elif val >= threshold_high:
            weight_factor = 1.2 + (val - threshold_high) / (threshold_high + 1) * 0.5  # máx ~1.7x
            weights[i] *= weight_factor

    return weights

# ==================== OPTIMIZACIÓN CON VALIDACIÓN CRUZADA TEMPORAL ====================
print("\nOptimizando modelo LightGBM (TimeSeriesSplit + pesos suavizados)...")

N_SPLITS = 4
tscv = TimeSeriesSplit(n_splits=N_SPLITS)


def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 3, 30),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42
    }

    fold_scores = []
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        val_weights = create_soft_weights(y_val)

        dtrain = lgb.Dataset(X_tr, y_tr)
        dval = lgb.Dataset(X_val, y_val, weight=val_weights, reference=dtrain)

        model_cv = lgb.train(
            params,
            dtrain,
            valid_sets=[dval],
            num_boost_round=3000,
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
        )

        y_pred = model_cv.predict(X_val, num_iteration=model_cv.best_iteration)

        general_mae = mean_absolute_error(y_val, y_pred)
        peak_mask = y_val > np.percentile(y_val, 80)
        peak_mae = mean_absolute_error(y_val[peak_mask], y_pred[peak_mask]) if np.sum(peak_mask) > 0 else general_mae

        fold_scores.append(0.7 * general_mae + 0.3 * peak_mae)

    return float(np.mean(fold_scores))


study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_params = study.best_params
print(f"\nMejores parámetros encontrados (promedio de {N_SPLITS} folds temporales):")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# ==================== ENTRENAMIENTO FINAL ====================
train_weights = create_soft_weights(y_train)

base_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42
}
model_params = {**base_params, **best_params}

dtrain_final = lgb.Dataset(X_train, y_train, weight=train_weights)
dtest_final = lgb.Dataset(X_test, y_test, reference=dtrain_final)

print("\nEntrenando modelo final...")
model = lgb.train(
    model_params,
    dtrain_final,
    valid_sets=[dtrain_final, dtest_final],
    valid_names=['train', 'test'],
    num_boost_round=5000,
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
)

print(f"\n[Diagnóstico] Mejor iteración (número de árboles usados): {model.best_iteration}")
if model.best_iteration is not None and model.best_iteration < 20:
    print("  Aviso: la mejor iteración es muy baja -> señal de underfitting / early stopping prematuro.")
    print("  Considera relajar la regularización o revisar si hay suficiente señal predictiva.")

# Predicciones (deshacer log1p si corresponde)
y_train_pred_raw = model.predict(X_train, num_iteration=model.best_iteration)
y_test_pred_raw = model.predict(X_test, num_iteration=model.best_iteration)

if USE_LOG_TARGET:
    y_train_pred = np.expm1(y_train_pred_raw)
    y_test_pred = np.expm1(y_test_pred_raw)
else:
    y_train_pred = y_train_pred_raw
    y_test_pred = y_test_pred_raw

y_train_eval = y_train_orig
y_test_eval = y_test_orig

# ==================== MÉTRICAS (siempre en escala original de casos) ====================
train_mae = mean_absolute_error(y_train_eval, y_train_pred)
test_mae = mean_absolute_error(y_test_eval, y_test_pred)
train_r2 = r2_score(y_train_eval, y_train_pred)
test_r2 = r2_score(y_test_eval, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_eval, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_eval, y_test_pred))

peak_mask_train = y_train_eval > np.percentile(y_train_eval, 80)
peak_mask_test = y_test_eval > np.percentile(y_test_eval, 80)
peak_mae_train = mean_absolute_error(y_train_eval[peak_mask_train], y_train_pred[peak_mask_train]) if np.sum(peak_mask_train) > 0 else np.nan
peak_mae_test = mean_absolute_error(y_test_eval[peak_mask_test], y_test_pred[peak_mask_test]) if np.sum(peak_mask_test) > 0 else np.nan

print("\n" + "="*60)
print("RESULTADOS DEL MODELO (METEO + DERIVADAS, SIN REZAGOS DE CASOS_DENGUE)")
print("="*60)
print(f"MAE Train: {train_mae:.2f}")
print(f"MAE Test: {test_mae:.2f}")
print(f"Peak MAE Train: {peak_mae_train:.2f}")
print(f"Peak MAE Test: {peak_mae_test:.2f}")
print(f"R² Train: {train_r2:.4f}")
print(f"R² Test: {test_r2:.4f}")
print("="*60)

# ==================== IMPORTANCIA DE FEATURES DEL MODELO FINAL (diagnóstico) ====================
lgb_importance = pd.DataFrame({
    'Feature': selected_features_rf,
    'Importance': model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 10 features según el modelo LightGBM final (importancia por ganancia):")
for i in range(min(10, len(lgb_importance))):
    print(f"  {i+1}. {lgb_importance.loc[i, 'Feature']}: {lgb_importance.loc[i, 'Importance']:.1f}")

fig_imp, ax_imp = plt.subplots(figsize=(8, 6))
top_imp = lgb_importance.head(15).iloc[::-1]
ax_imp.barh(top_imp['Feature'], top_imp['Importance'], color='teal')
ax_imp.set_title('Importancia de características (LightGBM final)')
ax_imp.set_xlabel('Importancia (gain)')
plt.tight_layout()
importance_plot_file = os.path.join(output_dir, 'importancia_features_lightgbm.png')
plt.savefig(importance_plot_file, dpi=200, bbox_inches='tight')
plt.close()
print(f"Gráfico de importancia guardado en: {importance_plot_file}")

# ==================== GRÁFICOS COMPARATIVOS ====================
print("\nGenerando gráficos...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Modelo LightGBM - Meteo + Variables Derivadas (sin rezagos de casos_dengue)',
             fontsize=13, fontweight='bold')

ax1.plot(fechas_train, y_train_eval, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax1.plot(fechas_train, y_train_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
peak_mask_train_plot = y_train_eval > np.percentile(y_train_eval, 80)
ax1.scatter(fechas_train[peak_mask_train_plot], y_train_eval[peak_mask_train_plot],
            color='gold', s=30, label='Picos (>80%)', zorder=5, alpha=0.8)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos Dengue')
ax1.set_title(f'Entrenamiento (2021-2025) - MAE: {train_mae:.2f}')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

ax2.plot(fechas_test, y_test_eval, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax2.plot(fechas_test, y_test_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
peak_mask_test_plot = y_test_eval > np.percentile(y_test_eval, 80)
ax2.scatter(fechas_test[peak_mask_test_plot], y_test_eval[peak_mask_test_plot],
            color='gold', s=30, label='Picos (>80%)', zorder=5, alpha=0.8)
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos Dengue')
ax2.set_title(f'Test (2026) - MAE: {test_mae:.2f}')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()

plot_file = os.path.join(output_dir, 'comparativa_entrenamiento_test_meteo_mejorado.png')
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Gráfico guardado en: {plot_file}")
plt.close()

# ==================== GUARDAR DATASET PROCESADO ====================
print("\nGuardando dataset procesado...")
processed_file = os.path.join(processed_dir, 'dataset_procesado_meteo_derivadas_rf.xlsx')

df_final = df_model[exclude_cols + [target_col] + selected_features_rf].copy()
df_final.to_excel(processed_file, index=False)
print(f"Dataset procesado guardado en: {processed_file}")

# ==================== RESULTADOS EN EXCEL ====================
print("\nGenerando Excel con resultados detallados...")
excel_file = os.path.join(output_dir, 'resultados_modelo_meteo_mejorado.xlsx')

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    metrics_df = pd.DataFrame({
        'Métrica': ['MAE', 'RMSE', 'R²', 'Peak MAE (80%)'],
        'Entrenamiento': [train_mae, train_rmse, train_r2, peak_mae_train],
        'Test': [test_mae, test_rmse, test_r2, peak_mae_test]
    })
    metrics_df.to_excel(writer, sheet_name='Métricas', index=False)

    pred_df = pd.DataFrame({
        'fecha': fechas_test,
        'año': años[test_mask],
        'semana_epi': semanas_test,
        'casos_reales': y_test_eval,
        'predicciones': y_test_pred,
        'error': np.abs(y_test_eval - y_test_pred),
        'es_pico': y_test_eval > np.percentile(y_test_eval, 80)
    })
    pred_df.to_excel(writer, sheet_name='Predicciones_Test', index=False)

    feature_importance_final = feature_importance[feature_importance['Feature'].isin(selected_features_rf)].copy()
    feature_importance_final.to_excel(writer, sheet_name='Features_RF', index=False)
    lgb_importance.to_excel(writer, sheet_name='Features_LightGBM', index=False)

    params_df = pd.DataFrame({
        'Parámetro': list(best_params.keys()) + ['best_iteration', 'use_log_target'],
        'Valor': [str(v) for v in best_params.values()] + [str(model.best_iteration), str(USE_LOG_TARGET)]
    })
    params_df.to_excel(writer, sheet_name='Parámetros', index=False)

    bins = [0, 5, 10, 20, 50, 100, 200]
    labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100-200']
    pred_df['rango_casos'] = pd.cut(pred_df['casos_reales'], bins=bins, labels=labels)

    error_analysis = pd.DataFrame()
    for label in labels:
        mask = pred_df['rango_casos'] == label
        if mask.sum() > 0:
            error_analysis.loc[label, 'Count'] = mask.sum()
            error_analysis.loc[label, 'MAE'] = pred_df[mask]['error'].mean()
            error_analysis.loc[label, 'Max_Error'] = pred_df[mask]['error'].max()
    error_analysis.to_excel(writer, sheet_name='Análisis_Errores')

print(f"Excel guardado en: {excel_file}")

# ==================== GUARDAR MODELO ====================
model_file = os.path.join(output_dir, 'modelo_meteo_mejorado_final.txt')
model.save_model(model_file)
print(f"Modelo guardado en: {model_file}")

# ==================== RESUMEN FINAL ====================
print("\n" + "="*70)
print("RESUMEN FINAL - MODELO METEO + DERIVADAS")
print("="*70)
print(f"✓ Predictores candidatos totales: {len(predictor_cols)}")
print(f"✓ Características seleccionadas por RF: {len(selected_features_rf)}")
print(f"✓ Target transformado con log1p: {USE_LOG_TARGET}")
print(f"✓ Mejor iteración LightGBM: {model.best_iteration}")
print(f"\n✓ MAE Entrenamiento: {train_mae:.2f}")
print(f"✓ MAE Test: {test_mae:.2f}")
print(f"✓ MAE Picos (Test): {peak_mae_test:.2f}")
print(f"✓ R² Test: {test_r2:.4f}")

print("\nEstrategias implementadas en esta versión:")
print("1. Anomalías estacionales por semana_epi (climatología calculada solo con años de entrenamiento)")
print("2. Acumulados de lluvia y días de lluvia (4, 8, 12 semanas)")
print("3. Rachas de condiciones favorables (temperatura y lluvia por encima de su climatología)")
print("4. Interacciones entre variables meteo (temp x hum_rel, temp_max x prec, índice vectorial simplificado)")
print("5. Agregados (promedio/min/max) de los rezagos de soi y sst a 6 y 12 semanas")
print("6. Pesos de picos suavizados (máx. ~2.5x en vez de 5x)")
print("7. Validación cruzada temporal (TimeSeriesSplit, 4 folds) en la optimización con Optuna")
print(f"8. Target transformado con log1p: {USE_LOG_TARGET}")
print("9. Diagnóstico de best_iteration e importancia de features del modelo final")

print("\nArchivos generados:")
print(f"  - Dataset procesado: {processed_file}")
print(f"  - Features seleccionados (RF): {features_file}")
print(f"  - Importancia features LightGBM: {importance_plot_file}")
print(f"  - Gráfico comparativo: {plot_file}")
print(f"  - Resultados Excel: {excel_file}")
print(f"  - Modelo guardado: {model_file}")
print("="*70)


Cargando datos...
Variables meteo base: 12
Rezagos meteo disponibles (1-12): 144

Creando variables climáticas derivadas...
Variables climáticas derivadas creadas: 47

Total de predictores candidatos (meteo + rezagos + derivadas, sin casos_dengue): 203
Registros disponibles tras eliminar NAs/infinitos: 270

Seleccionando características con Random Forest...


[I 2026-08-28 16:44:53,883] A new study created in memory with name: no-name-eedf5ee1-5d22-44f9-8ceb-41ba9fa63c6d


Top 30 características seleccionadas por RF:
  1. hum_esp_anomalia: 0.2453
  2. soi_min_12: 0.1286
  3. sst_max_12: 0.1177
  4. sst_min_6: 0.0336
  5. sst_anomalia: 0.0218
  6. hum_esp_lag_12: 0.0152
  7. dias_lluvia_acum_12: 0.0144
  8. hum_rel_anomalia: 0.0115
  9. temp_min: 0.0109
  10. hum_esp_lag_10: 0.0095
  11. hum_esp_lag_8: 0.0087
  12. temp_min_clima_semana: 0.0082
  13. sst: 0.0082
  14. hum_esp_lag_11: 0.0075
  15. prec_acum_12: 0.0060

Lista de atributos seleccionados guardada en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\atributos_seleccionados_rf.xlsx

Preparando datos finales...
Entrenamiento: 249 registros
Test: 21 registros

Optimizando modelo LightGBM (TimeSeriesSplit + pesos suavizados)...


Best trial: 0. Best value: 0.945007:   1%|▏         | 1/80 [00:00<00:11,  6.96it/s]

[I 2026-08-28 16:44:54,026] Trial 0 finished with value: 0.945006808584112 and parameters: {'num_leaves': 47, 'learning_rate': 0.06978211022792681, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8394633936788146, 'bagging_freq': 2, 'min_child_samples': 7, 'reg_alpha': 0.05808361216819946, 'reg_lambda': 0.8661761457749352, 'min_split_gain': 0.12022300234864176, 'max_depth': 10}. Best is trial 0 with value: 0.945006808584112.


...

[I 2026-08-28 16:45:14,646] Trial 74 finished with value: 0.943237545477801 and parameters: {'num_leaves': 37, 'learning_rate': 0.07645762717870905, 'feature_fraction': 0.9844298619582679, 'bagging_fraction': 0.9823621941102205, 'bagging_freq': 3, 'min_child_samples': 17, 'reg_alpha': 0.32204905375955367, 'reg_lambda': 0.29384331908781136, 'min_split_gain': 0.001264954049673318, 'max_depth': 4}. Best is trial 63 with value: 0.8844269709555481.
[I 2026-08-28 16:45:14,821] Trial 75 finished with value: 0.9037175734241616 and parameters: {'num_leaves': 26, 'learning_rate': 0.06567968189627515, 'feature_fraction': 0.9932958064195136, 'bagging_fraction': 0.9570079131239574, 'bagging_freq': 2, 'min_child_samples': 14, 'reg_alpha': 0.35724586555795296, 'reg_lambda': 0.1283790861863299, 'min_split_gain': 0.025377996539114468, 'max_depth': 3}. Best is trial 63 with value: 0.8844269709555481.


Best trial: 63. Best value: 0.884427:  98%|█████████▊| 78/80 [00:21<00:00,  5.07it/s]

[I 2026-08-28 16:45:15,013] Trial 76 finished with value: 0.8961740460174932 and parameters: {'num_leaves': 26, 'learning_rate': 0.06470332155619157, 'feature_fraction': 0.9937643891391487, 'bagging_fraction': 0.9616606996283708, 'bagging_freq': 3, 'min_child_samples': 14, 'reg_alpha': 0.34058206503177485, 'reg_lambda': 0.055343637801160556, 'min_split_gain': 0.02398167602276277, 'max_depth': 3}. Best is trial 63 with value: 0.8844269709555481.
[I 2026-08-28 16:45:15,203] Trial 77 finished with value: 0.8967334202447115 and parameters: {'num_leaves': 29, 'learning_rate': 0.058139491501488914, 'feature_fraction': 0.9987045598983896, 'bagging_fraction': 0.9812103554469775, 'bagging_freq': 4, 'min_child_samples': 13, 'reg_alpha': 0.24877905507935372, 'reg_lambda': 0.1595445622514518, 'min_split_gain': 0.02279024461351501, 'max_depth': 3}. Best is trial 63 with value: 0.8844269709555481.


Best trial: 63. Best value: 0.884427: 100%|██████████| 80/80 [00:21<00:00,  3.68it/s]


[I 2026-08-28 16:45:15,396] Trial 78 finished with value: 0.9038769580482784 and parameters: {'num_leaves': 30, 'learning_rate': 0.057517272353427934, 'feature_fraction': 0.9798575295056324, 'bagging_fraction': 0.963690966721936, 'bagging_freq': 4, 'min_child_samples': 13, 'reg_alpha': 0.2894869340215035, 'reg_lambda': 0.15148290295343272, 'min_split_gain': 0.024717637100907342, 'max_depth': 3}. Best is trial 63 with value: 0.8844269709555481.
[I 2026-08-28 16:45:15,590] Trial 79 finished with value: 0.9029869817618559 and parameters: {'num_leaves': 22, 'learning_rate': 0.07234089946471459, 'feature_fraction': 0.9991643218029792, 'bagging_fraction': 0.9879779660132306, 'bagging_freq': 5, 'min_child_samples': 15, 'reg_alpha': 0.24895097195828594, 'reg_lambda': 0.08774536593684967, 'min_split_gain': 0.00885555627927433, 'max_depth': 3}. Best is trial 63 with value: 0.8844269709555481.

Mejores parámetros encontrados (promedio de 4 folds temporales):
  num_leaves: 60
  learning_rate: 0.05

# 

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.model_selection import TimeSeriesSplit
import optuna

# ======================================================================
# CONFIGURACIÓN Y TOGGLES
# ======================================================================
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
processed_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

USE_LOG_TARGET = True

# Método de anomalía climática:
#   'rolling'  -> value - media móvil de las últimas N semanas (se adapta si el ciclo
#                 estacional del brote se desfasa de un año a otro; recomendado)
#   'calendar' -> value - promedio histórico de esa semana_epi (versión anterior,
#                 puede "anclar" al modelo a un calendario fijo)
ANOMALY_METHOD = 'rolling'
ROLLING_ANOMALY_WINDOW = 26

USE_PEAK_WEIGHTS = True          # poner en False para comparar sin pesos de picos
N_SELECTED_FEATURES = 15         # antes 30; menos features = menos varianza
N_ENSEMBLE_MODELS = 8            # bagging: nº de modelos LightGBM promediados
N_OPTUNA_TRIALS = 60
ADD_QUANTILE_BOUNDS = True       # agrega banda de incertidumbre (p10-p90)

# Años usados para el backtesting rápido de generalización (walk-forward)
BACKTEST_CUTOFFS = [2023, 2024, 2025, 2026]

# ======================================================================
# CARGA DE DATOS
# ======================================================================
print("Cargando datos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']

base_meteo_vars = [
    'temp', 'temp_max', 'temp_min',
    'hum_esp', 'hum_rel',
    'prec', 'dias_lluvia',
    'vel_vi', 'vel_vi_max', 'vel_vi_min',
    'soi', 'sst'
]
base_meteo_vars = [v for v in base_meteo_vars if v in df.columns]

lag_predictor_cols = []
for var in base_meteo_vars:
    for lag in range(1, 13):
        col = f'{var}_lag_{lag}'
        if col in df.columns:
            lag_predictor_cols.append(col)

print(f"Variables meteo base: {len(base_meteo_vars)}")
print(f"Rezagos meteo disponibles (1-12): {len(lag_predictor_cols)}")


# ======================================================================
# INGENIERÍA DE ATRIBUTOS CLIMÁTICOS (solo meteo, nunca casos_dengue)
# ======================================================================
def racha_consecutiva(condicion):
    racha = np.zeros(len(condicion), dtype=int)
    contador = 0
    for i, val in enumerate(condicion):
        contador = contador + 1 if val else 0
        racha[i] = contador
    return racha


def engineer_climate_features(df_in, train_years_mask, anomaly_method='rolling',
                               rolling_window=26):
    """
    Crea variables derivadas solo a partir de meteo/clima. Devuelve el dataframe
    enriquecido y la lista de columnas nuevas creadas.
    """
    df_eng = df_in.copy()
    engineered_cols = []

    # --- A. Anomalías ---
    for var in base_meteo_vars:
        col_anom = f'{var}_anomalia'
        if anomaly_method == 'calendar':
            clima = df_eng.loc[train_years_mask].groupby('semana_epi')[var].mean()
            media_global = df_eng.loc[train_years_mask, var].mean()
            col_clima = f'{var}_clima_semana'
            df_eng[col_clima] = df_eng['semana_epi'].map(clima).fillna(media_global)
            df_eng[col_anom] = df_eng[var] - df_eng[col_clima]
            engineered_cols += [col_anom, col_clima]
        else:  # 'rolling' -> no depende de un calendario fijo, se adapta en el tiempo
            roll_mean = df_eng[var].rolling(window=rolling_window, min_periods=1).mean()
            df_eng[col_anom] = df_eng[var] - roll_mean
            engineered_cols.append(col_anom)

    # --- B. Acumulados (lluvia y días de lluvia) ---
    if 'prec' in df_eng.columns:
        for window in [4, 8, 12]:
            col = f'prec_acum_{window}'
            df_eng[col] = df_eng['prec'].rolling(window=window, min_periods=1).sum()
            engineered_cols.append(col)

    if 'dias_lluvia' in df_eng.columns:
        for window in [4, 8, 12]:
            col = f'dias_lluvia_acum_{window}'
            df_eng[col] = df_eng['dias_lluvia'].rolling(window=window, min_periods=1).sum()
            engineered_cols.append(col)

    # --- C. Rachas de condiciones favorables ---
    if 'temp_anomalia' in df_eng.columns:
        df_eng['racha_temp_alta'] = racha_consecutiva((df_eng['temp_anomalia'] > 0).values)
        engineered_cols.append('racha_temp_alta')
    if 'prec_anomalia' in df_eng.columns:
        df_eng['racha_prec_alta'] = racha_consecutiva((df_eng['prec_anomalia'] > 0).values)
        engineered_cols.append('racha_prec_alta')

    # --- D. Interacciones ---
    if set(['temp', 'hum_rel']).issubset(df_eng.columns):
        df_eng['temp_x_hum_rel'] = df_eng['temp'] * df_eng['hum_rel']
        df_eng['indice_vectorial'] = (df_eng['temp'] * df_eng['hum_rel']) / 100.0
        engineered_cols += ['temp_x_hum_rel', 'indice_vectorial']
    if set(['temp_max', 'prec']).issubset(df_eng.columns):
        df_eng['temp_max_x_prec'] = df_eng['temp_max'] * df_eng['prec']
        engineered_cols.append('temp_max_x_prec')

    # --- E. Agregados de rezagos largos (SOI / SST) ---
    for var in ['soi', 'sst']:
        lag_cols_var = [f'{var}_lag_{lag}' for lag in range(1, 13) if f'{var}_lag_{lag}' in df_eng.columns]
        lag_cols_6 = [c for c in lag_cols_var if int(c.split('_')[-1]) <= 6]
        if lag_cols_6:
            df_eng[f'{var}_avg_6'] = df_eng[lag_cols_6].mean(axis=1)
            engineered_cols.append(f'{var}_avg_6')
        if lag_cols_var:
            df_eng[f'{var}_avg_12'] = df_eng[lag_cols_var].mean(axis=1)
            engineered_cols.append(f'{var}_avg_12')

    engineered_cols = list(dict.fromkeys(engineered_cols))
    return df_eng, engineered_cols


def create_soft_weights(y, peak_threshold=0.80, extreme_threshold=0.95):
    weights = np.ones_like(y, dtype=float)
    threshold_high = np.percentile(y, peak_threshold * 100)
    threshold_extreme = np.percentile(y, extreme_threshold * 100)
    for i, val in enumerate(y):
        if val >= threshold_extreme:
            weights[i] *= 1.8 + (val - threshold_extreme) / (threshold_extreme + 1) * 0.7
        elif val >= threshold_high:
            weights[i] *= 1.2 + (val - threshold_high) / (threshold_high + 1) * 0.5
    return weights


def select_features_rf(X, y, cols, n_selected):
    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    imp = pd.DataFrame({'Feature': cols, 'Importance': rf.feature_importances_}) \
        .sort_values('Importance', ascending=False).reset_index(drop=True)
    selected = imp.head(min(n_selected, len(cols)))['Feature'].tolist()
    return selected, imp


def train_single_lgb(X_tr, y_tr, X_val, y_val, params, use_weights, seed):
    p = dict(params)
    p['random_state'] = seed
    p['bagging_seed'] = seed
    weights = create_soft_weights(y_tr) if use_weights else None
    dtrain = lgb.Dataset(X_tr, y_tr, weight=weights)
    dval = lgb.Dataset(X_val, y_val, reference=dtrain)
    model = lgb.train(
        p, dtrain,
        valid_sets=[dval],
        num_boost_round=3000,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
    )
    return model


def train_ensemble(X_train, y_train, X_val, y_val, params, n_models, use_weights):
    """
    Bagging simple: cada modelo se entrena sobre una muestra bootstrap de
    (X_train, y_train), con una semilla distinta, y se detiene con early stopping
    usando (X_val, y_val) -- que NUNCA debe ser el test final.
    """
    models = []
    rng = np.random.RandomState(123)
    n = len(X_train)
    for i in range(n_models):
        idx = rng.choice(n, size=n, replace=True)
        X_boot, y_boot = X_train[idx], y_train[idx]
        model = train_single_lgb(X_boot, y_boot, X_val, y_val, params, use_weights, seed=42 + i)
        models.append(model)
    return models


def predict_ensemble(models, X, use_log_target):
    preds = np.array([m.predict(X, num_iteration=m.best_iteration) for m in models])
    if use_log_target:
        preds = np.expm1(preds)
    return preds.mean(axis=0), preds.std(axis=0)


def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    peak_mask = y_true > np.percentile(y_true, 80)
    peak_mae = mean_absolute_error(y_true[peak_mask], y_pred[peak_mask]) if peak_mask.sum() > 0 else np.nan
    return mae, rmse, r2, peak_mae


# ======================================================================
# PARTE 1: BACKTESTING WALK-FORWARD (diagnóstico de generalización)
# ======================================================================
# Antes de invertir tiempo en optimizar a fondo, se entrena/evalúa con parámetros
# fijos y razonablemente regularizados sobre varios cortes temporales, para saber
# si el gap train-test observado antes es sistemático o específico de 2026.
print("\n" + "="*70)
print("BACKTESTING WALK-FORWARD (parámetros fijos, diagnóstico rápido)")
print("="*70)

fixed_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 25,
    'learning_rate': 0.02,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'min_split_gain': 0.05,
    'max_depth': 5,
    'verbose': -1,
    'n_jobs': -1,
}

backtest_rows = []
for cutoff in BACKTEST_CUTOFFS:
    train_years_mask_bt = df['año'] < cutoff
    if train_years_mask_bt.sum() < 52:  # necesita al menos ~1 año de historia
        continue

    df_eng_bt, eng_cols_bt = engineer_climate_features(
        df, train_years_mask_bt, anomaly_method=ANOMALY_METHOD, rolling_window=ROLLING_ANOMALY_WINDOW
    )
    predictor_cols_bt = [c for c in (base_meteo_vars + lag_predictor_cols + eng_cols_bt) if c in df_eng_bt.columns]

    df_model_bt = df_eng_bt[exclude_cols + [target_col] + predictor_cols_bt].copy()
    df_model_bt = df_model_bt.replace([np.inf, -np.inf], np.nan).dropna(
        subset=predictor_cols_bt + [target_col]
    ).reset_index(drop=True)

    train_mask_bt = df_model_bt['año'] < cutoff
    test_mask_bt = df_model_bt['año'] == cutoff
    if train_mask_bt.sum() < 52 or test_mask_bt.sum() == 0:
        continue

    X_bt = df_model_bt[predictor_cols_bt].values
    y_bt_orig = df_model_bt[target_col].values
    y_bt = np.log1p(y_bt_orig) if USE_LOG_TARGET else y_bt_orig.copy()

    X_tr_bt, y_tr_bt = X_bt[train_mask_bt.values], y_bt[train_mask_bt.values]
    X_te_bt, y_te_bt_orig = X_bt[test_mask_bt.values], y_bt_orig[test_mask_bt.values]

    # RF selection dentro del propio backtest (usando solo datos hasta 'cutoff')
    sel_feats_bt, _ = select_features_rf(X_tr_bt, y_tr_bt, predictor_cols_bt, N_SELECTED_FEATURES)
    sel_idx_bt = [predictor_cols_bt.index(f) for f in sel_feats_bt]

    # Validación interna = últimas ~52 semanas de train, para early stopping
    n_val = min(52, max(10, int(len(X_tr_bt) * 0.15)))
    X_tr_inner, y_tr_inner = X_tr_bt[:-n_val, sel_idx_bt], y_tr_bt[:-n_val]
    X_val_inner, y_val_inner = X_tr_bt[-n_val:, sel_idx_bt], y_tr_bt[-n_val:]

    model_bt = train_single_lgb(X_tr_inner, y_tr_inner, X_val_inner, y_val_inner,
                                 fixed_params, USE_PEAK_WEIGHTS, seed=42)

    y_tr_pred_bt = model_bt.predict(X_tr_bt[:, sel_idx_bt], num_iteration=model_bt.best_iteration)
    y_te_pred_bt = model_bt.predict(X_te_bt[:, sel_idx_bt], num_iteration=model_bt.best_iteration)
    if USE_LOG_TARGET:
        y_tr_pred_bt = np.expm1(y_tr_pred_bt)
        y_te_pred_bt = np.expm1(y_te_pred_bt)

    y_tr_true_bt = y_bt_orig[train_mask_bt.values]
    mae_tr, rmse_tr, r2_tr, peak_mae_tr = compute_metrics(y_tr_true_bt, y_tr_pred_bt)
    mae_te, rmse_te, r2_te, peak_mae_te = compute_metrics(y_te_bt_orig, y_te_pred_bt)

    backtest_rows.append({
        'Corte (test=año)': cutoff,
        'N_train': int(train_mask_bt.sum()),
        'N_test': int(test_mask_bt.sum()),
        'MAE_train': mae_tr, 'MAE_test': mae_te,
        'PeakMAE_train': peak_mae_tr, 'PeakMAE_test': peak_mae_te,
        'R2_test': r2_te
    })
    print(f"  Corte {cutoff}: MAE train={mae_tr:.2f} | MAE test={mae_te:.2f} | "
          f"Peak MAE test={peak_mae_te:.2f} | R2 test={r2_te:.3f}")

backtest_df = pd.DataFrame(backtest_rows)
if len(backtest_df) > 0:
    print("\nPromedio del backtesting walk-forward:")
    print(backtest_df[['MAE_train', 'MAE_test', 'PeakMAE_train', 'PeakMAE_test', 'R2_test']].mean().to_string())
backtest_file = os.path.join(output_dir, 'backtesting_walkforward.xlsx')
backtest_df.to_excel(backtest_file, index=False)
print(f"Backtesting guardado en: {backtest_file}")


# ======================================================================
# PARTE 2: MODELO FINAL (2021-2025 -> 2026) CON OPTIMIZACIÓN Y ENSEMBLE
# ======================================================================
print("\n" + "="*70)
print("MODELO FINAL: entrenamiento 2021-2025, evaluación 2026")
print("="*70)

train_years_mask_final = (df['año'] >= 2021) & (df['año'] <= 2025)
df_eng, engineered_cols = engineer_climate_features(
    df, train_years_mask_final, anomaly_method=ANOMALY_METHOD, rolling_window=ROLLING_ANOMALY_WINDOW
)
predictor_cols = [c for c in (base_meteo_vars + lag_predictor_cols + engineered_cols) if c in df_eng.columns]
print(f"Predictores candidatos: {len(predictor_cols)}")

df_model = df_eng[exclude_cols + [target_col] + predictor_cols].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna(
    subset=predictor_cols + [target_col]
).reset_index(drop=True)
print(f"Registros disponibles: {len(df_model)}")

X_all = df_model[predictor_cols].values
y_all_orig = df_model[target_col].values
y_all = np.log1p(y_all_orig) if USE_LOG_TARGET else y_all_orig.copy()

años_all = df_model['año'].values
fechas_all = df_model['fecha'].values
semanas_all = df_model['semana_epi'].values

train_mask = (años_all >= 2021) & (años_all <= 2025)
test_mask = años_all == 2026

X_train_full, y_train_full, y_train_full_orig = X_all[train_mask], y_all[train_mask], y_all_orig[train_mask]
X_test, y_test_orig = X_all[test_mask], y_all_orig[test_mask]
fechas_train, fechas_test = fechas_all[train_mask], fechas_all[test_mask]

# --- Selección de features (limitada a N_SELECTED_FEATURES) ---
print(f"\nSeleccionando top {N_SELECTED_FEATURES} características con Random Forest...")
selected_features_rf, feature_importance = select_features_rf(
    X_train_full, y_train_full, predictor_cols, N_SELECTED_FEATURES
)
for i, feat in enumerate(selected_features_rf):
    imp_val = feature_importance.loc[feature_importance['Feature'] == feat, 'Importance'].values[0]
    print(f"  {i+1}. {feat}: {imp_val:.4f}")

features_file = os.path.join(output_dir, 'atributos_seleccionados_rf.xlsx')
feature_importance.to_excel(features_file, index=False)

sel_idx = [predictor_cols.index(f) for f in selected_features_rf]
X_train_sel = X_train_full[:, sel_idx]
X_test_sel = X_test[:, sel_idx]

# --- Split interno: validación = último año de train (2025), NUNCA el test (2026) ---
# Esto corrige un problema de la versión anterior, donde el early stopping se
# decidía mirando directamente el set de test (fuga de información hacia la
# selección del modelo).
val_year_mask = (df_model.loc[train_mask, 'año'] == 2025).values
X_opt_train, y_opt_train = X_train_sel[~val_year_mask], y_train_full[~val_year_mask]
X_opt_val, y_opt_val = X_train_sel[val_year_mask], y_train_full[val_year_mask]

print(f"\nEntrenamiento interno (2021-2024): {len(X_opt_train)} | Validación interna (2025): {len(X_opt_val)}")

# --- Optimización con Optuna: rangos más regularizados (menos libertad) ---
N_SPLITS = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS)


def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 10, 40),          # antes hasta 100
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.8),  # antes hasta 1.0
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.8),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 40),    # antes desde 3
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 3.0),                # antes desde 0
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 3.0),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 6),               # antes hasta 12
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42
    }

    fold_scores = []
    for tr_idx, val_idx in tscv.split(X_opt_train):
        X_tr, X_val = X_opt_train[tr_idx], X_opt_train[val_idx]
        y_tr, y_val = y_opt_train[tr_idx], y_opt_train[val_idx]

        model_cv = train_single_lgb(X_tr, y_tr, X_val, y_val, params, USE_PEAK_WEIGHTS, seed=42)
        y_pred = model_cv.predict(X_val, num_iteration=model_cv.best_iteration)

        mae = mean_absolute_error(y_val, y_pred)
        peak_mask = y_val > np.percentile(y_val, 80)
        peak_mae = mean_absolute_error(y_val[peak_mask], y_pred[peak_mask]) if peak_mask.sum() > 0 else mae
        fold_scores.append(0.7 * mae + 0.3 * peak_mae)

    return float(np.mean(fold_scores))


print("\nOptimizando hiperparámetros (rangos regularizados, TimeSeriesSplit)...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_params = study.best_params
print("\nMejores parámetros:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

base_params = {
    'objective': 'regression', 'metric': 'mae', 'boosting_type': 'gbdt',
    'verbose': -1, 'n_jobs': -1
}
final_params = {**base_params, **best_params}

# --- Entrenamiento del ensemble final (bagging), validando con 2025, evaluando en 2026 ---
print(f"\nEntrenando ensemble de {N_ENSEMBLE_MODELS} modelos (bagging)...")
ensemble_models = train_ensemble(
    X_opt_train, y_opt_train, X_opt_val, y_opt_val,
    final_params, N_ENSEMBLE_MODELS, USE_PEAK_WEIGHTS
)

best_iters = [m.best_iteration for m in ensemble_models]
print(f"Mejores iteraciones por modelo del ensemble: {best_iters}")

y_train_pred, _ = predict_ensemble(ensemble_models, X_train_sel, USE_LOG_TARGET)
y_test_pred, y_test_std = predict_ensemble(ensemble_models, X_test_sel, USE_LOG_TARGET)

# --- Métricas ---
train_mae, train_rmse, train_r2, peak_mae_train = compute_metrics(y_train_full_orig, y_train_pred)
test_mae, test_rmse, test_r2, peak_mae_test = compute_metrics(y_test_orig, y_test_pred)

print("\n" + "="*60)
print("RESULTADOS DEL MODELO FINAL (ENSEMBLE)")
print("="*60)
print(f"MAE Train: {train_mae:.2f} | MAE Test: {test_mae:.2f}")
print(f"Peak MAE Train: {peak_mae_train:.2f} | Peak MAE Test: {peak_mae_test:.2f}")
print(f"R² Train: {train_r2:.4f} | R² Test: {test_r2:.4f}")
print("="*60)

# --- Bandas de incertidumbre opcionales (regresión cuantílica) ---
if ADD_QUANTILE_BOUNDS:
    print("\nEntrenando modelos de cuantiles (p10 / p90) para banda de incertidumbre...")
    q_params_low = {**base_params, **best_params, 'objective': 'quantile', 'alpha': 0.1, 'metric': 'quantile'}
    q_params_high = {**base_params, **best_params, 'objective': 'quantile', 'alpha': 0.9, 'metric': 'quantile'}
    model_q_low = train_single_lgb(X_opt_train, y_opt_train, X_opt_val, y_opt_val, q_params_low, False, seed=7)
    model_q_high = train_single_lgb(X_opt_train, y_opt_train, X_opt_val, y_opt_val, q_params_high, False, seed=8)

    y_test_q_low = model_q_low.predict(X_test_sel, num_iteration=model_q_low.best_iteration)
    y_test_q_high = model_q_high.predict(X_test_sel, num_iteration=model_q_high.best_iteration)
    if USE_LOG_TARGET:
        y_test_q_low = np.expm1(y_test_q_low)
        y_test_q_high = np.expm1(y_test_q_high)
else:
    y_test_q_low = y_test_q_high = None

# ======================================================================
# GRÁFICOS
# ======================================================================
print("\nGenerando gráficos...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Modelo LightGBM (Ensemble) - Meteo + Derivadas, regularizado', fontsize=13, fontweight='bold')

ax1.plot(fechas_train, y_train_full_orig, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax1.plot(fechas_train, y_train_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
peak_mask_tr_plot = y_train_full_orig > np.percentile(y_train_full_orig, 80)
ax1.scatter(fechas_train[peak_mask_tr_plot], y_train_full_orig[peak_mask_tr_plot],
            color='gold', s=25, label='Picos (>80%)', zorder=5, alpha=0.8)
ax1.set_title(f'Entrenamiento (2021-2025) - MAE: {train_mae:.2f}')
ax1.set_xlabel('Fecha'); ax1.set_ylabel('Casos Dengue')
ax1.legend(loc='upper left', fontsize=8); ax1.grid(True, alpha=0.3); ax1.tick_params(axis='x', rotation=45)

ax2.plot(fechas_test, y_test_orig, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax2.plot(fechas_test, y_test_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
if ADD_QUANTILE_BOUNDS:
    ax2.fill_between(fechas_test, y_test_q_low, y_test_q_high, color='red', alpha=0.15, label='Banda p10-p90')
peak_mask_te_plot = y_test_orig > np.percentile(y_test_orig, 80)
ax2.scatter(fechas_test[peak_mask_te_plot], y_test_orig[peak_mask_te_plot],
            color='gold', s=25, label='Picos (>80%)', zorder=5, alpha=0.8)
ax2.set_title(f'Test (2026) - MAE: {test_mae:.2f}')
ax2.set_xlabel('Fecha'); ax2.set_ylabel('Casos Dengue')
ax2.legend(loc='upper left', fontsize=8); ax2.grid(True, alpha=0.3); ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plot_file = os.path.join(output_dir, 'comparativa_entrenamiento_test_ensemble.png')
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"Gráfico guardado en: {plot_file}")

# Gráfico del backtesting walk-forward
if len(backtest_df) > 0:
    fig_bt, ax_bt = plt.subplots(figsize=(9, 5))
    ax_bt.plot(backtest_df['Corte (test=año)'], backtest_df['MAE_train'], 'o-', label='MAE train')
    ax_bt.plot(backtest_df['Corte (test=año)'], backtest_df['MAE_test'], 'o-', label='MAE test')
    ax_bt.plot(backtest_df['Corte (test=año)'], backtest_df['PeakMAE_test'], 'o--', label='Peak MAE test')
    ax_bt.set_xlabel('Año usado como test (walk-forward)')
    ax_bt.set_ylabel('MAE')
    ax_bt.set_title('Backtesting walk-forward: ¿el gap train-test es sistemático?')
    ax_bt.legend(); ax_bt.grid(True, alpha=0.3)
    plt.tight_layout()
    backtest_plot_file = os.path.join(output_dir, 'backtesting_walkforward.png')
    plt.savefig(backtest_plot_file, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"Gráfico de backtesting guardado en: {backtest_plot_file}")

# ======================================================================
# GUARDAR RESULTADOS
# ======================================================================
processed_file = os.path.join(processed_dir, 'dataset_procesado_final.xlsx')
df_model[exclude_cols + [target_col] + selected_features_rf].to_excel(processed_file, index=False)

excel_file = os.path.join(output_dir, 'resultados_modelo_final.xlsx')
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    metrics_df = pd.DataFrame({
        'Métrica': ['MAE', 'RMSE', 'R²', 'Peak MAE (80%)'],
        'Entrenamiento': [train_mae, train_rmse, train_r2, peak_mae_train],
        'Test': [test_mae, test_rmse, test_r2, peak_mae_test]
    })
    metrics_df.to_excel(writer, sheet_name='Métricas', index=False)

    pred_df = pd.DataFrame({
        'fecha': fechas_test,
        'año': años_all[test_mask],
        'semana_epi': semanas_all[test_mask],
        'casos_reales': y_test_orig,
        'prediccion_media': y_test_pred,
        'prediccion_std_ensemble': y_test_std,
        'error': np.abs(y_test_orig - y_test_pred),
        'es_pico': y_test_orig > np.percentile(y_test_orig, 80)
    })
    if ADD_QUANTILE_BOUNDS:
        pred_df['p10'] = y_test_q_low
        pred_df['p90'] = y_test_q_high
    pred_df.to_excel(writer, sheet_name='Predicciones_Test', index=False)

    feature_importance.to_excel(writer, sheet_name='Features_RF', index=False)
    backtest_df.to_excel(writer, sheet_name='Backtesting_WalkForward', index=False)

    params_df = pd.DataFrame({
        'Parámetro': list(best_params.keys()) + ['use_log_target', 'use_peak_weights',
                                                   'n_selected_features', 'n_ensemble_models',
                                                   'anomaly_method'],
        'Valor': [str(v) for v in best_params.values()] + [
            str(USE_LOG_TARGET), str(USE_PEAK_WEIGHTS), str(N_SELECTED_FEATURES),
            str(N_ENSEMBLE_MODELS), ANOMALY_METHOD
        ]
    })
    params_df.to_excel(writer, sheet_name='Parámetros', index=False)

print(f"Excel guardado en: {excel_file}")

for i, m in enumerate(ensemble_models):
    m.save_model(os.path.join(output_dir, f'modelo_ensemble_{i}.txt'))
print(f"Modelos del ensemble guardados en: {output_dir}")

# ======================================================================
# RESUMEN FINAL
# ======================================================================
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)
print(f"✓ Predictores candidatos: {len(predictor_cols)} | Seleccionados (RF): {len(selected_features_rf)}")
print(f"✓ Método de anomalía: {ANOMALY_METHOD} | Pesos de picos: {USE_PEAK_WEIGHTS}")
print(f"✓ Ensemble de {N_ENSEMBLE_MODELS} modelos | Target log1p: {USE_LOG_TARGET}")
print(f"\n✓ MAE Train: {train_mae:.2f} | MAE Test: {test_mae:.2f}")
print(f"✓ Peak MAE Train: {peak_mae_train:.2f} | Peak MAE Test: {peak_mae_test:.2f}")
print(f"✓ R² Test: {test_r2:.4f}")
if len(backtest_df) > 0:
    print(f"\n✓ Backtesting walk-forward (promedio {len(backtest_df)} cortes):")
    print(f"  MAE test promedio: {backtest_df['MAE_test'].mean():.2f}")
    print(f"  Peak MAE test promedio: {backtest_df['PeakMAE_test'].mean():.2f}")

print("\nCambios clave respecto a la versión anterior:")
print("1. Backtesting walk-forward (2023,2024,2025,2026) para saber si el gap train-test es sistemático")
print("2. N_SELECTED_FEATURES reducido de 30 a", N_SELECTED_FEATURES)
print("3. Rangos de Optuna más regularizados (max_depth<=6, num_leaves<=40, reg_alpha/lambda>=0.1, etc.)")
print("4. Anomalías con ventana móvil ('rolling') en vez de calendario fijo, para evitar desfases estacionales")
print("5. Validación para early stopping = último año de TRAIN (2025), nunca el test (2026, sin fuga)")
print("6. Ensemble/bagging de", N_ENSEMBLE_MODELS, "modelos para reducir varianza")
print(f"7. Banda de incertidumbre p10-p90 (regresión cuantílica): {ADD_QUANTILE_BOUNDS}")
print("\nArchivos generados en:", output_dir)
print("="*70)


Cargando datos...
Variables meteo base: 12
Rezagos meteo disponibles (1-12): 144

BACKTESTING WALK-FORWARD (parámetros fijos, diagnóstico rápido)
  Corte 2023: MAE train=2.43 | MAE test=12.59 | Peak MAE test=24.13 | R2 test=-3.139
  Corte 2024: MAE train=2.83 | MAE test=31.67 | Peak MAE test=58.96 | R2 test=-2.833
  Corte 2025: MAE train=7.64 | MAE test=36.48 | Peak MAE test=76.04 | R2 test=-1.806
  Corte 2026: MAE train=11.94 | MAE test=16.09 | Peak MAE test=31.60 | R2 test=-2.102

Promedio del backtesting walk-forward:
MAE_train         6.210277
MAE_test         24.208366
PeakMAE_train    19.953509
PeakMAE_test     47.683299
R2_test          -2.470018
Backtesting guardado en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\backtesting_walkforward.xlsx

MODELO FINAL: entrenamiento 2021-2025, evaluación 2026
Predictores candidatos: 183
Registros disponibles: 270

Seleccionando top 15 características con Random Forest...


[I 2026-08-28 20:55:36,791] A new study created in memory with name: no-name-e1c2c664-2f04-46c6-bcae-9837fa7f47f7


  1. dias_lluvia_acum_12: 0.1661
  2. soi_avg_12: 0.1271
  3. vel_vi_max: 0.0439
  4. vel_vi_max_lag_1: 0.0365
  5. hum_esp_lag_5: 0.0325
  6. hum_esp_lag_10: 0.0273
  7. hum_esp_lag_9: 0.0264
  8. hum_esp_lag_6: 0.0255
  9. sst_avg_12: 0.0254
  10. hum_esp_lag_7: 0.0235
  11. vel_vi_max_lag_4: 0.0231
  12. hum_esp_lag_8: 0.0210
  13. sst: 0.0186
  14. sst_anomalia: 0.0177
  15. hum_esp_lag_4: 0.0156

Entrenamiento interno (2021-2024): 196 | Validación interna (2025): 53

Optimizando hiperparámetros (rangos regularizados, TimeSeriesSplit)...


Best trial: 0. Best value: 1.02232:   3%|▎         | 2/60 [00:00<00:11,  5.19it/s]

[I 2026-08-28 20:55:37,012] Trial 0 finished with value: 1.022320814317172 and parameters: {'num_leaves': 21, 'learning_rate': 0.044635901521768134, 'feature_fraction': 0.7195981825434216, 'bagging_fraction': 0.679597545259111, 'bagging_freq': 2, 'min_child_samples': 14, 'reg_alpha': 0.2684424752877784, 'reg_lambda': 2.611910822747312, 'min_split_gain': 0.18033450352296262, 'max_depth': 5}. Best is trial 0 with value: 1.022320814317172.
[I 2026-08-28 20:55:37,185] Trial 1 finished with value: 1.0421155543090854 and parameters: {'num_leaves': 10, 'learning_rate': 0.04665303012212833, 'feature_fraction': 0.7497327922401266, 'bagging_fraction': 0.5637017332034828, 'bagging_freq': 2, 'min_child_samples': 15, 'reg_alpha': 0.9823025045826593, 'reg_lambda': 1.6217936517334899, 'min_split_gain': 0.12958350559263473, 'max_depth': 4}. Best is trial 0 with value: 1.022320814317172.


...

[I 2026-08-28 20:55:57,784] Trial 56 finished with value: 1.043317267124878 and parameters: {'num_leaves': 20, 'learning_rate': 0.01378464952996892, 'feature_fraction': 0.5884269250477616, 'bagging_fraction': 0.6255066576185636, 'bagging_freq': 3, 'min_child_samples': 13, 'reg_alpha': 1.9666617587567181, 'reg_lambda': 0.6882883086053085, 'min_split_gain': 0.08915916940626051, 'max_depth': 3}. Best is trial 33 with value: 0.9635670289274684.
[I 2026-08-28 20:55:57,934] Trial 57 finished with value: 1.1967661710210962 and parameters: {'num_leaves': 30, 'learning_rate': 0.02003679808995008, 'feature_fraction': 0.6556681946539025, 'bagging_fraction': 0.7301941590546591, 'bagging_freq': 1, 'min_child_samples': 35, 'reg_alpha': 1.7827699668719261, 'reg_lambda': 2.5551438467469163, 'min_split_gain': 0.2109925422339639, 'max_depth': 4}. Best is trial 33 with value: 0.9635670289274684.


Best trial: 58. Best value: 0.960403:  98%|█████████▊| 59/60 [00:21<00:00,  3.36it/s]

[I 2026-08-28 20:55:58,260] Trial 58 finished with value: 0.9604028708800062 and parameters: {'num_leaves': 26, 'learning_rate': 0.01858853865099239, 'feature_fraction': 0.6486202270166433, 'bagging_fraction': 0.7499015466588194, 'bagging_freq': 1, 'min_child_samples': 11, 'reg_alpha': 1.3219547014488684, 'reg_lambda': 2.9287179953619553, 'min_split_gain': 0.06439565969293293, 'max_depth': 4}. Best is trial 58 with value: 0.9604028708800062.


Best trial: 59. Best value: 0.957321: 100%|██████████| 60/60 [00:21<00:00,  2.76it/s]


[I 2026-08-28 20:55:58,563] Trial 59 finished with value: 0.9573211390648364 and parameters: {'num_leaves': 26, 'learning_rate': 0.016877125433943378, 'feature_fraction': 0.6810460041388892, 'bagging_fraction': 0.74359298020693, 'bagging_freq': 1, 'min_child_samples': 10, 'reg_alpha': 1.3283214865792465, 'reg_lambda': 2.9147210292387613, 'min_split_gain': 0.11737782695596533, 'max_depth': 4}. Best is trial 59 with value: 0.9573211390648364.

Mejores parámetros:
  num_leaves: 26
  learning_rate: 0.016877125433943378
  feature_fraction: 0.6810460041388892
  bagging_fraction: 0.74359298020693
  bagging_freq: 1
  min_child_samples: 10
  reg_alpha: 1.3283214865792465
  reg_lambda: 2.9147210292387613
  min_split_gain: 0.11737782695596533
  max_depth: 4

Entrenando ensemble de 8 modelos (bagging)...
Mejores iteraciones por modelo del ensemble: [618, 498, 521, 277, 1320, 160, 329, 759]

RESULTADOS DEL MODELO FINAL (ENSEMBLE)
MAE Train: 9.42 | MAE Test: 18.07
Peak MAE Train: 33.04 | Peak MAE Te